In [15]:
# Install chromadb to store and search vector embeddings
!pip install -q chromadb
# Install the modern Google GenAI SDK
!pip install -q google-genai
# Import the client for Google Gemini
from google import genai
# Import chromadb for civic complaint vector storage
import chromadb
# Print setup confirmation to verify everything works
print("Setup ready")


Setup ready


In [16]:
# @title
# # List containing 6 dummy environmental complaints matching the required format
# documents = [
#     "Issue: Garbage dumping in public park. Authority: DUMMY-AUTHORITY-1. Submit via: DUMMY-PORTAL. Rule: REGULATION-A. Escalate to: DEPT-HEAD-X.",
#     "Issue: Illegal waste burning behind warehouse. Authority: DUMMY-AUTHORITY-2. Submit via: DUMMY-EMAIL. Rule: REGULATION-B. Escalate to: DEPT-HEAD-Y.",
#     "Issue: Water leakage on Main Street. Authority: DUMMY-AUTHORITY-3. Submit via: DUMMY-HOTLINE. Rule: REGULATION-C. Escalate to: DEPT-HEAD-Z.",
#     "Issue: Excessive plastic waste accumulation. Authority: DUMMY-AUTHORITY-4. Submit via: DUMMY-APP. Rule: REGULATION-D. Escalate to: DEPT-HEAD-W.",
#     "Issue: Toxic waste disposal in storm drain. Authority: DUMMY-AUTHORITY-5. Submit via: DUMMY-SMS. Rule: REGULATION-E. Escalate to: DEPT-HEAD-V.",
#     "Issue: Broken water main flooding alleyway. Authority: DUMMY-AUTHORITY-6. Submit via: DUMMY-FORM. Rule: REGULATION-F. Escalate to: DEPT-HEAD-U."
# ]

# # Matching IDs for each of the dummy complaints
# ids = [f"complaint_{i}" for i in range(1, 7)]

# # Print the number of items created to verify the setup
# print(f"Number of documents: {len(documents)}")
# print(f"Number of IDs: {len(ids)}")

In [17]:
# Verified Knowledge Base for Jaipur Civic & Environmental Complaints (Hari Awaaz)
# Every fact is extracted strictly from verified official Rajasthan government portals & PDFs.

documents = [
    # 1. Garbage Dumping & Municipal Solid Waste (Source: jaipurmc.org & JMC Officer Directory PDF)
    "Issue: Garbage dumping and uncollected municipal solid waste (Keywords: kachra, kooda, solid waste, overflowing dustbin, street litter, safai, garbage pile). Authority: Nagar Nigam Jaipur (Jaipur Municipal Corporation) - Deputy Commissioner (Health) & Executive Engineer (Solid Waste Management). Submit via: JMC Call Centre / Public Grievance Helpline 0141-2747400 (08:00 AM to 10:00 PM, 7 days), 'Jaipur Samadhan' Android mobile app, or online at http://jaipurmc.org (About Us -> Online Services -> Register Your Complaint). Rule: Solid Waste Management Rules, 2026 & Rajasthan Municipalities Act, 2009 (Section 226). Escalate to: Commissioner JMC (0141-2740510, commissioner.jmc@rajasthan.gov.in) or JMC Control Room (0141-2742900). Source: http://jaipurmc.org/Presentation/PublicNotice/FAQs.aspx (Verified: 23-11-2026).",

    # 2. Construction & Demolition (C&D) Waste Dumping (Source: RSPCB CandDWaste.html & ComplaintPage.html)
    "Issue: Construction and demolition waste dumping or malba accumulation (Keywords: C&D waste, malba, construction debris, building waste, rubble, patthar, mitti). Authority: Rajasthan State Pollution Control Board (RSPCB). Submit via: Rajasthan Sampark Portal (https://sampark.rajasthan.gov.in) under Public Grievance with location, nature of pollution, and photos/videos. Rule: Construction and Demolition Waste Management Rules, 2016. Escalate to: Rajasthan Sampark Helpline 181 or Department of Public Grievances (0141-2922825, ds.rpg@rajasthan.gov.in). Source: https://environment.rajasthan.gov.in/content/environment/en/rajasthan-state-pollution-control-board/ComplaintPage.html (Verified: 23-11-2026).",

    # 3. Waste / Open Burning (Source: RSPCB ComplaintPage.html & sampark.rajasthan.gov.in)
    "Issue: Waste and open garbage burning (Keywords: kachra jalana, kooda jalana, patte jalana, dry leaves burning, plastic burning, garbage burning, smoke, dhuan, aag lagana). Authority: Rajasthan State Pollution Control Board (RSPCB). Submit via: Rajasthan Sampark Portal (https://sampark.rajasthan.gov.in) under Public Grievance with location, nature of pollution, and photos/videos. Rule: Solid Waste Management Rules, 2026 (Rule 21 - Polluter Pays Principle) & Air (Prevention and Control of Pollution) Act, 1981. Escalate to: Rajasthan Sampark Toll-Free Helpline 181 (0141-2922272, 2921646) or Department of Public Grievances (ds.rpg@rajasthan.gov.in). Source: https://environment.rajasthan.gov.in/content/environment/en/rajasthan-state-pollution-control-board/ComplaintPage.html (Verified: 23-11-2026).",

    # 4. Water Supply Leakage & Pipeline Burst (Source: PHED Control Room page & District Control Room PDF)
    "Issue: Drinking water supply leakage, pipeline burst, or water wastage (Keywords: paani leak, pipeline burst, broken water pipe, water wastage, nal ka paani behna, drinking water supply, jal). Authority: Public Health Engineering Department (PHED), Government of Rajasthan - Jaipur District Control Room. Submit via: Jaipur District Control Room Phone 0141-2706624 (Incharge: Smt. Neelima Agrawal, 9636580186 / 8279100526) or PHED State Control Room 0141-2222585. Rule: not available. Escalate to: Chief Engineer (Urban & NRW), PHED Jaipur (0141-2222053, rj_cehq@nic.in) or Rajasthan Sampark Helpline 181. Source: https://sampark.rajasthan.gov.in (Verified: 24-11-2026).",

    # 5. Sewer Overflow & Choked Storm Drains (Source: jaipurmc.org & JMC Officer Directory PDF)
    "Issue: Sewer overflow, choked drains, and dirty water stagnation (Keywords: naali jaam, sewer overflow, gatar, drainage blockage, dirty water stagnation, sewage leak). Authority: Nagar Nigam Jaipur (Jaipur Municipal Corporation) - Superintending Engineer (Sewer Project) / Executive Engineer (Liquid Waste Management). Submit via: JMC Call Centre 0141-2747400 (08:00 AM to 10:00 PM), 'Jaipur Samadhan' app, or http://jaipurmc.org (Online Services -> Register Your Complaint). Rule: Rajasthan Municipalities Act, 2009 (Chapter XI - Drainage and Sewerage) & Water (Prevention and Control of Pollution) Act, 1974. Escalate to: JMC Control Room (0141-2742900) or Commissioner JMC (commissioner.jmc@rajasthan.gov.in). Source: http://jaipurmc.org/Presentation/Others/ConctactUs.aspx (Verified: 23-11-2026).",

    # 6. Tree Cutting & Forest Protection (Source: forest.rajasthan.gov.in & sampark.rajasthan.gov.in)
    "Issue: Illegal tree cutting, felling, or damage to forest trees (Keywords: ped katna, tree cutting, illegal tree felling, branch cutting, jungle, forest destruction). Authority: Forest Department, Government of Rajasthan. Submit via: Rajasthan Sampark Portal (https://rajasthan.gov.in -> Parivad Panjikaran) or Toll-Free Helpline 181 (0141-2922272, 2921646). Rule: The Rajasthan Forest Act, 1953 and Rajasthan Forest Policy 2023. Escalate to: Department of Public Grievances, Secretariat Jaipur (0141-2922825, 0141-2385077, ds.rpg@rajasthan.gov.in). Source: https://sampark.rajasthan.gov.in (Verified: 24-11-2026).",

    # 7. Biodiversity & Heritage Grove Destruction (Source: Rajasthan State Biodiversity Board Grievance & Functions pages)
    "Issue: Destruction of local biodiversity, heritage sites, sacred groves, or wildlife habitats (Keywords: biodiversity damage, sacred grove, heritage trees, flora fauna harm, wildlife disturbance). Authority: Rajasthan State Biodiversity Board, Jaipur (Member Secretary I.F.S. Shree Roop Narayan Meena / Chief Manager Technical R.F.S. Shri Mukesh Kumar Tiwari). Submit via: Phone 0141-2362188, 0141-2362088, 9899954865 or Email rajsbb@rajasthan.gov.in / rajasthansbb@gmail.com (Address: Arawali Bhawan, First Floor, Jhalana Institutional Area, Jaipur 302004). Rule: Biological Diversity Act, 2002 and Rajasthan Biological Diversity Rules, 2010. Escalate to: Rajasthan Sampark Portal (https://rajasthan.gov.in) or Helpline 181. Source: https://sampark.rajasthan.gov.in (Verified: 24-11-2026).",

    # 8. Noise & Air Pollution (Source: RSPCB ComplaintPage.html & sampark.rajasthan.gov.in)
    "Issue: Noise pollution and industrial or vehicular air pollution (Keywords: tez awaaz, shor, loudspeaker, DJ noise, late night music, generator noise, factory smoke, dhuan, air pollution, dust). Authority: Rajasthan State Pollution Control Board (RSPCB). Submit via: Rajasthan Sampark Portal (https://rajasthan.gov.in) under Public Grievance with location, nature of pollution, and photos/videos. Rule: Noise Pollution (Regulation and Control) Rules, 2000 (Rule 5) & Air (Prevention and Control of Pollution) Act, 1981.. Escalate to: Rajasthan Sampark Toll-Free Helpline 181 (0141-2922272, 2921646) or Department of Public Grievances (ds.rpg@rajasthan.gov.in). Source: https://rajasthan.gov.in (Verified: 23-11-2026).",

    # 9. General Unresolved Civic Grievance Escalation (Source: sampark.rajasthan.gov.in/ContactUs)
    "Issue: General civic or environmental grievance escalation when no action is taken (Keywords: unresolved complaint, no action, sunwai nahi hui, escalation, shikayat, pending complaint, parivad panjikaran). Authority: Department of Public Grievances, Government of Rajasthan (Rajasthan Sampark). Submit via: Rajasthan Sampark Portal (https://rajasthan.gov.in -> Parivad Panjikaran), Toll-Free Helpline 181, Call Center (0141-2922272, 2921646), Tech Helpdesk (0141-2922293), or Email rajsampark@rajasthan.gov.in. Rule: not available. Escalate to: Director of Public Grievances & Ex-Officio Special Secretary (Shree Harsh Sawan Sukha, IAS) - Room No. 7220, Food Building, 2nd Floor, Secretariat, Jaipur (0141-2922825, 0141-2385077, ds.rpg@rajasthan.gov.in). Source: https://rajasthan.gov.in/ContactUs (Verified: 24-11-2026)."
]

# Automatically generate matching IDs for all 9 documents
ids = [f"complaint_{i}" for i in range(1, len(documents) + 1)]

print(f"Number of verified documents: {len(documents)}")
print(f"Number of IDs: {len(ids)}")


Number of verified documents: 9
Number of IDs: 9


In [18]:
import chromadb

# Initialize an ephemeral (in-memory) ChromaDB client
chroma_client = chromadb.Client()

# Create or get a collection named 'civic_complaints'
# ChromaDB's default embedding function will automatically embed the text documents
collection = chroma_client.get_or_create_collection("civic_complaints")

# Add our dummy documents and their corresponding IDs to the collection
collection.add(
    documents=documents,
    ids=ids
)

print("ChromaDB collection initialized and populated with documents.")

ChromaDB collection initialized and populated with documents.


In [19]:
def retrieve(complaint):
    """
    Queries the ChromaDB collection with the user's complaint
    and returns the top 2 matching documents.
    """
    # Query the collection for the top 2 closest matches
    results = collection.query(
        query_texts=[complaint],
        n_results=2
    )
    # Extract and return the documents from the nested results structure
    return results['documents'][0]

# Test the retrieve function with the user's query
test_query = "kachra jalate hain park ke paas"
retrieved_results = retrieve(test_query)

print(f"Query: '{test_query}'\n")
print("Top 2 Matching Documents:")
for idx, doc in enumerate(retrieved_results, 1):
    print(f"{idx}. {doc}")

Query: 'kachra jalate hain park ke paas'

Top 2 Matching Documents:
1. Issue: Waste and open garbage burning (Keywords: kachra jalana, kooda jalana, patte jalana, dry leaves burning, plastic burning, garbage burning, smoke, dhuan, aag lagana). Authority: Rajasthan State Pollution Control Board (RSPCB). Submit via: Rajasthan Sampark Portal (https://sampark.rajasthan.gov.in) under Public Grievance with location, nature of pollution, and photos/videos. Rule: Solid Waste Management Rules, 2026 (Rule 21 - Polluter Pays Principle) & Air (Prevention and Control of Pollution) Act, 1981. Escalate to: Rajasthan Sampark Toll-Free Helpline 181 (0141-2922272, 2921646) or Department of Public Grievances (ds.rpg@rajasthan.gov.in). Source: https://environment.rajasthan.gov.in/content/environment/en/rajasthan-state-pollution-control-board/ComplaintPage.html (Verified: 23-11-2026).
2. Issue: Construction and demolition waste dumping or malba accumulation (Keywords: C&D waste, malba, construction debris,

In [20]:
# for m in client.models.list():
#     if 'gemini' in m.name:
#         print(m.name)


In [21]:
import time
from google.colab import userdata
from google import genai

api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

# List primary + fallback models from your Step 1 output
MODELS = ['gemini-3.8-flash', 'gemini-3.5-flash-lite', 'gemini-3.8-pro']

def safe_generate(contents, config=None, retries=3):
    for model_name in MODELS:
        for attempt in range(retries):
            try:
                return client.models.generate_content(
                    model=model_name,
                    contents=contents,
                    config=config
                )
            except Exception as e:
                err_str = str(e)
                if "503" in err_str or "UNAVAILABLE" in err_str or "429" in err_str or "RESOURCE_EXHAUSTED" in err_str:
                    time.sleep(2 ** attempt)  # Wait 1s, 2s, 4s before retrying
                    continue
                raise e  # Raise non-503 errors (like 404 or invalid key) immediately
    raise RuntimeError("All fallback models are temporarily busy. Try again in a moment.")

# Quick test
response = safe_generate('Say "Gemini setup with fallback is ready!"')
print(response.text)


Gemini setup with fallback is ready!


In [23]:
import json
from google.genai import types

def classify(complaint: str) -> dict:
    """
    Classifies a civic complaint using the Gemini API with automatic retry/fallback
    and parses the JSON response.
    """
    prompt = f"""Role: You are a civic complaint assistant for Jaipur.
Task: classify the complaint into exactly one of: Garbage dumping, Waste/open burning, Water leakage/wastage, Tree cutting, Noise/air pollution.
Context: the complaint may be in Hindi, English or Hinglish.
Constraints: answer only with a category and urgency (low/medium/high). If it fits none, answer 'Not covered'. Do not guess authorities or contacts.
Output format: JSON with keys category and urgency.

Complaint: {complaint} """

    raw_text = ""
    try:
        # Call safe_generate instead of client.models.generate_content
        response = safe_generate(
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.1,
                response_mime_type="application/json"
            )
        )
        raw_text = response.text.strip()

        # Safely parse the JSON response
        parsed_json = json.loads(raw_text)
        return parsed_json

    except json.JSONDecodeError:
        return {"error": "Invalid JSON response from model", "raw_response": raw_text}
    except Exception as e:
        return {"error": f"An error occurred: {str(e)}"}

# Test the function with sample complaints
print(classify("Roz subah park ke paas log kachra jalate hain, dhuan ghar tak aata hai."))
print(classify("Main road par pipeline phat gayi hai, kal se paani beh raha hai."))
print(classify("What is the capital of France?"))


{'category': 'Waste/open burning', 'urgency': 'medium'}
{'category': 'Water leakage/wastage', 'urgency': 'high'}
{'category': 'Not covered', 'urgency': 'low'}


In [24]:
def draft_complaint(complaint: str, category: str, retrieved_entry: str, temperature: float = 0.2) -> str:
    """
    Drafts a structured formal civic complaint letter using ONLY verified facts
    in retrieved_entry. Falls back to 'not available' for missing fields.
    """
    prompt = f"""
Role: You are 'Hari Awaaz', an official civic complaint assistant for Jaipur, Rajasthan.
Task: Write a clear, structured formal complaint letter and citizen action summary based ONLY on the 'Retrieved Entry' below.

STRICT GROUNDING RULES:
1. Use ONLY the facts (Authority, Submit via, Rule, Escalate to, Source) present in the 'Retrieved Entry'.
2. Do NOT invent, guess, or add any phone numbers, emails, portal links, officer names, or legal Acts that are not explicitly written in 'Retrieved Entry'.
3. If any field (such as Rule or Escalate to) states 'not available' in the 'Retrieved Entry', write exactly "not available" for that item.
4. Language Matching: If 'User Complaint' is written in Devanagari Hindi script, write the formal "letter" in polite, official Hindi (Devanagari script), keeping all phone numbers, emails, and URLs verbatim. If 'User Complaint' is written in English or Hinglish (Roman script), write the "letter" in clear, formal English.

REQUIRED OUTPUT FORMAT:
**To:** [Authority from Retrieved Entry]
**Subject:** Formal Complaint Regarding {category} in Jaipur

**Respected Sir/Madam,**
[2-3 sentences formally describing the citizen's complaint based on 'User Complaint' and requesting prompt inspection and resolution.]

**Applicable Legal Rule / Act:** [Rule from Retrieved Entry]
**Where & How to Submit:** [Submit via from Retrieved Entry]
**Escalation Path (If Unresolved):** [Escalate to from Retrieved Entry]
**Official Reference Source:** [Source URL from Retrieved Entry]

User Complaint: {complaint}
Category: {category}
Retrieved Entry: {retrieved_entry}
"""

    try:
        response = safe_generate(
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=temperature
            )
        )
        return response.text.strip()
    except Exception as e:
        return f"An error occurred while drafting the complaint: {str(e)}"


In [27]:
def handle_complaint(text: str) -> str:
    """
    Orchestrates the civic complaint process:
    1. Checks if the complaint text is empty.
    2. Classifies the complaint using Gemini.
    3. Checks for API errors or 'Not covered' category.
    4. Queries ChromaDB to find the top 2 matching reference documents.
    5. Drafts a formal letter using the retrieved reference details.
    """
    # 1. Check if empty
    if not text or not text.strip():
        return "Please provide the details of your complaint so we can assist you."

    try:
        # 2. Call classify
        classification = classify(text)

        # Surface API/rate-limit errors instead of hiding them as 'Not covered'
        if "error" in classification:
            return f"Temporary service issue while classifying complaint: {classification['error']}"

        category = str(classification.get("category", "Not covered")).strip()

        # 3. Check if covered (case-insensitive)
        if category.lower() == "not covered":
            return "Thank you for reaching out. However, this issue is not covered under Jaipur civic complaints."

        # 4. Retrieve top 2 matching documents from ChromaDB for richer RAG context
        query_result = collection.query(query_texts=[text], n_results=2)

        if query_result and "documents" in query_result and query_result["documents"][0]:
            retrieved_entry = "\n\n".join(query_result["documents"][0])
        else:
            retrieved_entry = "No matching reference database entry found."

        # 5. Draft formal complaint letter
        letter = draft_complaint(text, category, retrieved_entry)
        return f"Category: {category}\nUrgency: {classification.get('urgency', 'medium')}\n\n{letter}"

    except Exception as e:
        print(f"An error occurred while handling the complaint: {e}")
        return "We encountered an unexpected error while processing your request. Please try again later."

# Test with three sample complaints
print("=== TEST 1: Covered Complaint ===")
print(handle_complaint("Hamare mohalle mein pipeline leak ho gayi hai, paani sadak par beh raha hai."))
print("\n=== TEST 2: Not Covered Complaint ===")
print(handle_complaint("Can you help me book a movie ticket for tonight?"))
print("\n=== TEST 3: Empty Complaint ===")
print(handle_complaint("   "))


=== TEST 1: Covered Complaint ===
Category: Water leakage/wastage
Urgency: medium

**To:** Public Health Engineering Department (PHED), Government of Rajasthan - Jaipur District Control Room
**Subject:** Formal Complaint Regarding Water leakage/wastage in Jaipur

**Respected Sir/Madam,**
Hamare mohalle mein pipeline leak ho gayi hai, aur paani sadak par beh raha hai jisase paani ki kafi barbadi ho rahi hai. Prarthana hai ki is samasya ki jald se jald jaanch karwayi jaye aur iska uchit samadhan kiya jaye.

**Applicable Legal Rule / Act:** not available
**Where & How to Submit:** Jaipur District Control Room Phone 0141-2706624 (Incharge: Smt. Neelima Agrawal, 9636580186 / 8279100526) or PHED State Control Room 0141-2222585.
**Escalation Path (If Unresolved):** Chief Engineer (Urban & NRW), PHED Jaipur (0141-2222053, rj_cehq@nic.in) or Rajasthan Sampark Helpline 181.
**Official Reference Source:** https://sampark.rajasthan.gov.in

=== TEST 2: Not Covered Complaint ===
Thank you for reachi

In [28]:
import time

# Use the real, verified Nagar Nigam Jaipur (JMC) entry from Cell 2
sample_retrieved = documents[0]
sample_complaint = "People are throwing plastic waste in the children's park daily and no one is cleaning it."
sample_category = "Garbage dumping"

temperatures = [0.0, 0.5, 1.0]

for temp in temperatures:
    print(f"=== DRAFT COMPLAINT AT TEMPERATURE {temp} ===")
    draft = draft_complaint(
        complaint=sample_complaint,
        category=sample_category,
        retrieved_entry=sample_retrieved,
        temperature=temp
    )
    print(draft)
    print("\n" + "="*40 + "\n")
    time.sleep(3)  # Pause briefly to stay safely under Free Tier RPM limits


=== DRAFT COMPLAINT AT TEMPERATURE 0.0 ===
**To:** Nagar Nigam Jaipur (Jaipur Municipal Corporation) - Deputy Commissioner (Health) & Executive Engineer (Solid Waste Management)
**Subject:** Formal Complaint Regarding Garbage dumping in Jaipur

**Respected Sir/Madam,**
I am writing to formally report an issue regarding people throwing plastic waste in the children's park daily, while no cleaning is being conducted. This unauthorized garbage dumping is creating unhygienic conditions and inconveniencing local residents and children. Therefore, I kindly request a prompt inspection and necessary resolution of this matter at the earliest.

**Applicable Legal Rule / Act:** Solid Waste Management Rules, 2026 & Rajasthan Municipalities Act, 2009 (Section 226)
**Where & How to Submit:** JMC Call Centre / Public Grievance Helpline 0141-2747400 (08:00 AM to 10:00 PM, 7 days), 'Jaipur Samadhan' Android mobile app, or online at http://jaipurmc.org (About Us -> Online Services -> Register Your Compl

In [29]:
!pip install -q pycloudflared flask-cors
import json
from datetime import datetime, timezone, timedelta
from flask import Flask, request, jsonify
from flask_cors import CORS
from google.genai import types
import threading, time
from pycloudflared import try_cloudflare

app = Flask(__name__)
CORS(app)

def build_structured_response(text: str, language: str = "English", location: str = "", timestamp: str = "") -> dict:
    # 1. Classify complaint
    classification = classify(text)
    if "error" in classification:
        return {"error": classification["error"]}

    category = str(classification.get("category", "Not covered")).strip()
    urgency = str(classification.get("urgency", "medium")).strip()

    if category.lower() == "not covered":
        return {"error": "This issue is not covered under Jaipur civic & environmental complaints."}

    # 2. Retrieve top 2 matching documents from ChromaDB
    query_result = collection.query(query_texts=[text], n_results=2)
    retrieved_entry = "\n\n".join(query_result["documents"][0]) if (query_result and query_result.get("documents")) else ""

    # 3. Set letter language instruction
    lang_clean = language.strip().lower()
    if "hindi" in lang_clean or "हिन्दी" in lang_clean:
        lang_instruction = "Write the 'letter' field completely in polite, official Hindi (Devanagari script), while keeping all phone numbers, emails, and URLs verbatim. Close with [आपका नाम] and [मोबाइल नंबर]."
    else:
        lang_instruction = "Write the 'letter' field in clear, formal English. Close with [Your Name] and [Your Contact Number]."

    location_context = location if location else "Not provided (use placeholder [Exact Landmark / Ward, Jaipur])"

    # 4. Generate structured JSON matching your frontend's showResult(data) schema
    prompt = f"""
Role: You are 'Hari Awaaz', an official civic complaint assistant for Jaipur, Rajasthan.
Task: Extract the verified details from 'Retrieved Entry' and draft a formal complaint letter based ONLY on those facts.

STRICT RULES:
1. Use ONLY facts explicitly written in 'Retrieved Entry'. Do NOT guess or invent phone numbers, emails, links, or legal rules.
2. If any field (like rule) is 'not available' in 'Retrieved Entry', write exactly "not available".
3. Language Requirement: {lang_instruction}
4. Timestamp & Location Grounding: Include the 'Report Timestamp' clearly at the top of the letter (e.g., Date & Time: {timestamp}). Include 'Location / Landmark' clearly in the letter; if marked "Not provided", insert the placeholder [Exact Landmark / Ward, Jaipur].
5. Return valid JSON with these exact keys:
   - "authority": exact department/authority name
   - "how_to_submit": exact helpline, app, or portal instructions
   - "rule": exact Act/Rule or "not available"
   - "source": exact source URL from Retrieved Entry
   - "letter": complete formal complaint letter addressed to the authority (plain text with line breaks, no markdown asterisks)

User Complaint: {text}
Location / Landmark: {location_context}
Report Timestamp: {timestamp}
Category: {category}
Retrieved Entry: {retrieved_entry}
"""
    res = safe_generate(
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2,
            response_mime_type="application/json"
        )
    )
    parsed = json.loads(res.text.strip())
    parsed["category"] = category
    parsed["urgency"] = urgency
    return parsed

@app.route('/api/complaint', methods=['POST'])
def process_complaint():
    try:
        data = request.get_json(force=True)
        user_text = data.get('complaint', '').strip()
        target_language = data.get('language', 'English').strip()
        location_text = data.get('location', '').strip()

        ist = timezone(timedelta(hours=5, minutes=30))
        default_ts = datetime.now(ist).strftime("%d %b %Y, %I:%M %p IST")
        report_timestamp = data.get('timestamp', '').strip() or default_ts

        if not user_text:
            return jsonify({"error": "Empty complaint text."}), 400
        result = build_structured_response(user_text, target_language, location_text, report_timestamp)
        return jsonify(result)
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# Run Flask on port 5004 and open a fresh Cloudflare Tunnel
threading.Thread(
    target=lambda: app.run(host='0.0.0.0', port=5004, use_reloader=False),
    daemon=True
).start()
time.sleep(2)

tunnel_url = try_cloudflare(port=5004)
print(f"\n✅ NEW LIVE API URL:\n{tunnel_url.tunnel}/api/complaint\n")


 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5004
 * Running on http://172.28.0.12:5004
INFO:werkzeug:Press CTRL+C to quit


 * Running on https://cult-baker-gene-eng.trycloudflare.com
 * Traffic stats available on http://127.0.0.1:20241/metrics

✅ NEW LIVE API URL:
https://cult-baker-gene-eng.trycloudflare.com/api/complaint

